# Trades File Acceptance Full `<1B>` Closeout

Este notebook rehace la lectura de `trades` sobre el cierre full final `57f/full_clean_fast_same_schema`.

A diferencia de una version reducida de closeout, aqui se revisan otra vez las seis capas de la metodologia de `05`, pero distinguiendo entre dos tipos de evidencia:

- capas `1`, `2` y `6`: se leen desde artefactos full ya materializados en `57f`
- capas `3`, `4` y `5`: se reconstruyen para lectura visual y resumen operativo desde una muestra determinista tomada de los `raw_metrics_shards` full del mismo `57f`

Eso permite mantener la misma narrativa de seis capas sin fingir que todos los res?menes intermedios quedaron persistidos en disco como tablas full independientes.


## Contexto

`05_trades_file_acceptance_audit.ipynb` sigue siendo el notebook metodologico de referencia.

Este `06` ya no discute si el run full existe: lo da por cerrado y usa `57f` como referencia canonica. Su objetivo es leer el cierre final en clave de seis capas, no solo mostrar el agregado final de politica.


In [ ]:
import json
import runpy
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

VIEW_58D = Path(
    r"C:/TSIS_Data/02_backtest_SmallCaps/data_auditoria_polygon/00_data_certification/auditoria/trades/cell_code/58d_trades_file_acceptance_view_lt1b_full_clean_fast_same_schema.py"
)

view = runpy.run_path(str(VIEW_58D))
globals().update(view)

manifest = load_manifest()
progress = load_progress()
arts = load_all_artifacts()
layer1 = arts["layer1_integrity_summary"]
layer2 = arts["layer2_coverage_summary_full"]
layer6 = arts["layer6_policy_summary_full"]
sample_index = arts["sample_index"]
policy_examples = load_policy_examples_top(top_n_per_label=12)

show_manifest(manifest, progress)


## Helpers para capas 3, 4 y 5

En `57f` no quedaron materializados como tablas full separadas `layer3_tape_quality_summary.parquet`, `layer4_reference_consistency_summary.parquet` ni `layer5_severity_real_summary.parquet`.

Pero sus columnas si viven en `raw_metrics_shards`. Por eso este notebook reconstruye esas tres capas desde una muestra determinista cross-shard, suficiente para lectura visual y resumen operativo, manteniendo la base final de `57f`.


In [ ]:
RAW_DIR = Path(
    r"C:/TSIS_Data/02_backtest_SmallCaps/runs/backtest/trades_v2_materialized/trades_current_cd_merged/root_cause_exports/file_acceptance_cache_lt1b_full_clean_fast_same_schema/raw_metrics_shards"
)

SAMPLE_COLS = [
    "file",
    "ticker",
    "date",
    "acceptance_label",
    "regular_trade_pct",
    "prepost_trade_pct",
    "off_session_trade_pct",
    "has_1m_reference",
    "has_daily_reference",
    "condition_combo_nunique",
    "duplicate_exact_ratio_pct_raw",
    "max_trades_same_timestamp_raw",
    "outside_daily_regular_pct",
    "outside_1m_regular_pct",
    "outside_daily_volume_pct",
    "outside_minutes_pct_active",
    "longest_outside_run_minutes",
    "top_outside_minute_trade_share_pct",
]


def load_full_raw_metrics_sample(sample_per_shard: int = 1500, seed: int = 42) -> pd.DataFrame:
    chunks = []
    for i, shard in enumerate(sorted(RAW_DIR.glob("raw_metrics_*.parquet"))):
        df = pd.read_parquet(shard, columns=SAMPLE_COLS)
        if len(df) > sample_per_shard:
            df = df.sample(sample_per_shard, random_state=seed + i)
        chunks.append(df)
    if not chunks:
        return pd.DataFrame(columns=SAMPLE_COLS)
    return pd.concat(chunks, ignore_index=True)


def _summary_df(rows: list[tuple[str, float]]) -> pd.DataFrame:
    return pd.DataFrame([{"metric": k, "value": float(v) if pd.notna(v) else np.nan} for k, v in rows])


def build_layer2_full_compare_table(raw_metrics: pd.DataFrame, sample_index: pd.DataFrame) -> pd.DataFrame:
    legacy_off = pd.to_numeric(sample_index.get("m.off_session_trade_pct"), errors="coerce")
    raw_off = pd.to_numeric(raw_metrics.get("off_session_trade_pct"), errors="coerce")
    one_m = pd.to_numeric(raw_metrics.get("has_1m_reference"), errors="coerce")
    daily = pd.to_numeric(raw_metrics.get("has_daily_reference"), errors="coerce")
    combos = pd.to_numeric(raw_metrics.get("condition_combo_nunique"), errors="coerce")
    rows = [
        ("sample_files_raw_sample", len(raw_metrics)),
        ("legacy_files_with_off_session_pct_gt_0", 100 * (legacy_off > 0).mean()),
        ("raw_files_with_off_session_pct_gt_0", 100 * (raw_off > 0).mean()),
        ("legacy_median_off_session_trade_pct", legacy_off.median()),
        ("raw_median_off_session_trade_pct", raw_off.median()),
        ("raw_files_with_1m_reference_pct", 100 * one_m.mean()),
        ("raw_files_with_daily_reference_pct", 100 * daily.mean()),
        ("median_condition_combo_nunique", combos.median()),
        ("p95_condition_combo_nunique", combos.quantile(0.95)),
    ]
    return _summary_df(rows)


def plot_layer2_observability_full(raw_metrics: pd.DataFrame, sample_index: pd.DataFrame) -> None:
    legacy_off = pd.to_numeric(sample_index.get("m.off_session_trade_pct"), errors="coerce")
    raw_off = pd.to_numeric(raw_metrics.get("off_session_trade_pct"), errors="coerce")
    combos = pd.to_numeric(raw_metrics.get("condition_combo_nunique"), errors="coerce")
    with plt.rc_context(PLOT_RC):
        fig, axes = plt.subplots(1, 3, figsize=(14.5, 4))
        axes[0].hist(legacy_off.dropna(), bins=20, color="#4C72B0")
        axes[0].set_title("Capa 2 | Off-session heredado")
        axes[0].set_xlabel("pct trades off-session")
        axes[1].hist(raw_off.dropna(), bins=20, color="#55A868")
        axes[1].set_title("Capa 2 | Off-session raw recomputado")
        axes[1].set_xlabel("pct trades off-session")
        axes[2].hist(combos.dropna(), bins=15, color="#C44E52")
        axes[2].set_title("Capa 2 | Diversidad de conditions")
        axes[2].set_xlabel("n conditions_key distintas por file")
        plt.tight_layout()
        plt.show()


def build_layer3_summary_from_sample(raw_metrics: pd.DataFrame) -> pd.DataFrame:
    dup = pd.to_numeric(raw_metrics.get("duplicate_exact_ratio_pct_raw"), errors="coerce")
    burst = pd.to_numeric(raw_metrics.get("max_trades_same_timestamp_raw"), errors="coerce")
    return _summary_df([
        ("median_duplicate_exact_ratio_pct_raw", dup.median()),
        ("p95_duplicate_exact_ratio_pct_raw", dup.quantile(0.95)),
        ("median_max_trades_same_timestamp_raw", burst.median()),
    ])


def build_layer4_summary_from_sample(raw_metrics: pd.DataFrame) -> pd.DataFrame:
    daily = pd.to_numeric(raw_metrics.get("outside_daily_regular_pct"), errors="coerce")
    m1 = pd.to_numeric(raw_metrics.get("outside_1m_regular_pct"), errors="coerce")
    vol = pd.to_numeric(raw_metrics.get("outside_daily_volume_pct"), errors="coerce")
    return _summary_df([
        ("median_outside_daily_regular_pct", daily.median()),
        ("median_outside_1m_regular_pct", m1.median()),
        ("median_outside_daily_volume_pct", vol.median()),
    ])


def build_layer5_summary_from_sample(raw_metrics: pd.DataFrame) -> pd.DataFrame:
    active = pd.to_numeric(raw_metrics.get("outside_minutes_pct_active"), errors="coerce")
    run = pd.to_numeric(raw_metrics.get("longest_outside_run_minutes"), errors="coerce")
    top = pd.to_numeric(raw_metrics.get("top_outside_minute_trade_share_pct"), errors="coerce")
    return _summary_df([
        ("median_outside_minutes_pct_active", active.median()),
        ("median_longest_outside_run_minutes", run.median()),
        ("median_top_outside_minute_trade_share_pct", top.median()),
    ])


def build_metric_profile_table(raw_metrics: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        s = pd.to_numeric(raw_metrics.get(col), errors="coerce")
        rows.append({
            "metric": col,
            "median": float(s.median()) if s.notna().any() else np.nan,
            "p90": float(s.quantile(0.90)) if s.notna().any() else np.nan,
            "p95": float(s.quantile(0.95)) if s.notna().any() else np.nan,
            "share_eq_0_pct": float((s == 0).mean() * 100),
            "share_eq_100_pct": float((s == 100).mean() * 100),
        })
    return pd.DataFrame(rows)


raw_sample = load_full_raw_metrics_sample(sample_per_shard=1500, seed=42)
layer3 = build_layer3_summary_from_sample(raw_sample)
layer4 = build_layer4_summary_from_sample(raw_sample)
layer5 = build_layer5_summary_from_sample(raw_sample)
layer2_sample_desc = build_layer2_full_compare_table(raw_sample, sample_index)
layer3_profile = build_metric_profile_table(raw_sample, [
    "duplicate_exact_ratio_pct_raw",
    "max_trades_same_timestamp_raw",
])
layer4_profile = build_metric_profile_table(raw_sample, [
    "outside_daily_regular_pct",
    "outside_1m_regular_pct",
    "outside_daily_volume_pct",
])
layer5_profile = build_metric_profile_table(raw_sample, [
    "outside_minutes_pct_active",
    "longest_outside_run_minutes",
    "top_outside_minute_trade_share_pct",
])

display(Markdown(
    f"Muestra determinista para capas 3-5: **{len(raw_sample):,}** files repartidos sobre los **95** shards full de `57f`."
))


## Primera capa

La primera capa sigue respondiendo la misma pregunta binaria: si el file es fisicamente utilizable antes de entrar en microestructura o comparabilidad.


In [ ]:
display(Markdown(summary_layer1(layer1)))
display(Markdown(explain_layer1_metrics(layer1)))
display(layer1)
plot_layer1_integrity(layer1)


## Segunda capa

Aqui conviene separar tres cosas:

- cobertura full materializada del universo `<1B>`
- se?al heredada de `sample_index`, que sirve como contraste historico pero no como verdad raw
- lectura recomputada desde `57f`, que es la base util para closeout


In [ ]:
display(Markdown(summary_layer2_full(layer2)))
display(layer2)
plot_layer2_full(layer2)

display(Markdown("### Segunda capa | Comparativa heredado vs raw recomputado"))
display(layer2_sample_desc)
plot_layer2_sessions(raw_sample)
plot_layer2_observability_full(raw_sample, sample_index)


## Tercera capa

La tercera capa vuelve a leer suciedad mecanica del tape. En este closeout full se reconstruye desde muestra determinista cross-shard del `57f`.


In [ ]:
display(Markdown(summary_layer3(layer3)))
display(layer3)
display(layer3_profile)
plot_layer3_quality(raw_sample)


## Cuarta capa

La cuarta capa mide masa real del desvio contra referencias `daily` y `1m`. Aqui tambien se reconstruye desde la muestra determinista tomada del cierre full.


In [ ]:
display(Markdown(summary_layer4(layer4)))
display(layer4)
display(layer4_profile)
plot_layer4_consistency(raw_sample)


## Quinta capa

La quinta capa distingue spike puntual frente a problema persistente. Igual que en `05`, aqui importa la persistencia y concentracion del outside, no solo que exista.


In [ ]:
display(Markdown(summary_layer5(layer5)))
display(layer5)
display(layer5_profile)
plot_layer5_severity(raw_sample)


## Sexta capa

La sexta capa es la salida operativa del run full final. Aqui si usamos el agregado materializado exacto de `57f`.


In [ ]:
display(Markdown(summary_layer6(layer6)))
display(layer6)
plot_layer6_policy(layer6)

display(Markdown("### Ejemplos top por label"))
display(policy_examples)


## Cierre

Este notebook deja alineadas las seis capas sobre el cierre final `57f` y mejora dos cosas frente a una lectura naive del full:

- separa mejor escala visual y semantica en `capa 1` y `capa 2`
- hace explicitas las colas extremas de `0/100` en `capas 4-5`, para no confundirlas con ruido de plotting

Con esto `06` vuelve a cubrir la arquitectura metodologica de `05`, pero con una lectura full `<1B>` mas limpia y mas defendible.
